In [1]:
from sqlalchemy import create_engine
from sqlalchemy import select
from mc_postgres_db.operations import set_data
from mc_postgres_db.models import ProviderAssetMarket, Provider, Asset, ProviderAsset
from dotenv import load_dotenv
import os

load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

engine = create_engine(POSTGRES_URL)

with engine.connect() as conn:
    stmt = select(Provider).where(Provider.name == "Kraken")
    result = conn.execute(stmt).first()
    print(result)

(1, 1, 'Kraken', 'Kraken is a globally recognized cryptocurrency exchange that offers secure and reliable trading for a wide range of digital assets. Founded in 2011,  ... (112 characters truncated) ... nown for its strong security measures and regulatory compliance, Kraken is one of the longest-running and most trusted platforms in the crypto space.', 'KRAKEN', None, None, None, True, datetime.datetime(2025, 6, 18, 0, 29, 0, 882210), datetime.datetime(2025, 6, 18, 0, 29, 0, 882210))


In [5]:
from sqlalchemy import select
import pandas as pd
from sqlalchemy.orm import Session
from mc_postgres_db.models import Asset

with Session(engine) as session:
    stmt = select(Provider).where(Provider.name == "Kraken")
    provider = session.execute(stmt).scalar_one()

    print(provider)

    stmt = select(Asset).where(Asset.name == "XLM")
    base_asset = session.execute(stmt).scalar_one()

    print(base_asset)

    stmt = select(Asset).where(Asset.name == "USD")
    usd = session.execute(stmt).scalar_one()

    print(usd)

    path = f"/Users/glynfinck/Downloads/Kraken_OHLCVT/{base_asset.name}{usd.name}_1.csv"

    # Read CSV without headers
    df = pd.read_csv(path, header=None)

    # Set meaningful column names for OHLCV data
    df.columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume', 'trade_count']

    # Format the timestamp to be a datetime object
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
    df["provider_id"] = provider.id
    df["from_asset_id"] = usd.id
    df["to_asset_id"] = xrp.id
    df["open"] = df["open"].astype(float)
    df["high"] = df["high"].astype(float)
    df["low"] = df["low"].astype(float)
    df["close"] = df["close"].astype(float)
    df["volume"] = df["volume"].astype(float)
    df["trade_count"] = df["trade_count"].astype(int)

    df.drop(columns=["trade_count"], inplace=True)

# Display the first few rows of the dataframe
df

Provider(1, Kraken)
Asset(9, XLM)
Asset(2, USD)


,timestamp,open,high,low,close,volume,provider_id,from_asset_id,to_asset_id
0,2017-01-17 20:17:00,0.002251,0.002251,0.002251,0.002251,145.578065,1,2,9
1,2017-01-20 07:04:00,0.002250,0.002250,0.002250,0.002250,5172.843978,1,2,9
2,2017-01-20 07:51:00,0.002250,0.002250,0.002250,0.002250,403.146493,1,2,9
3,2017-01-20 08:01:00,0.002250,0.002250,0.002250,0.002250,554.005575,1,2,9
4,2017-01-20 08:11:00,0.002250,0.002250,0.002250,0.002250,577.479140,1,2,9
...,...,...,...,...,...,...,...,...,...
1422631,2025-03-31 23:50:00,0.263488,0.263488,0.263488,0.263488,707.494480,1,2,9
1422632,2025-03-31 23:51:00,0.263522,0.263522,0.263522,0.263522,526.811890,1,2,9
1422633,2025-03-31 23:53:00,0.263815,0.263816,0.263815,0.263816,2857.963800,1,2,9
1422634,2025-03-31 23:55:00,0.263913,0.263913,0.263913,0.263913,100.350140,1,2,9


In [6]:
from tqdm.notebook import tqdm

chunk_size = 100000
total_rows = len(df)

print(f"Processing {total_rows} rows in batches of {chunk_size}")

for i in tqdm(range(0, total_rows, chunk_size)):
    chunk = df.iloc[i:i+chunk_size]
    batch_num = (i // chunk_size) + 1
    total_batches = (total_rows + chunk_size - 1) // chunk_size
    try:
        set_data(engine, ProviderAssetMarket.__tablename__, chunk, operation_type="upsert")
    except Exception as e:
        print(f"Error processing batch {batch_num}: {e}")
        # You might want to break here or continue depending on your needs
        break

print("Batch processing complete!")

Processing 1422636 rows in batches of 100000


  0%|          | 0/15 [00:00<?, ?it/s]

Upserting 100000 row(s) to provider_asset_market
Upserting 100000 row(s) to provider_asset_market
Upserting 100000 row(s) to provider_asset_market
Upserting 100000 row(s) to provider_asset_market
Upserting 100000 row(s) to provider_asset_market
Upserting 100000 row(s) to provider_asset_market
Upserting 100000 row(s) to provider_asset_market
Upserting 100000 row(s) to provider_asset_market
Upserting 100000 row(s) to provider_asset_market
Upserting 100000 row(s) to provider_asset_market
Upserting 100000 row(s) to provider_asset_market
Upserting 100000 row(s) to provider_asset_market
Upserting 100000 row(s) to provider_asset_market
Upserting 100000 row(s) to provider_asset_market
Upserting 22636 row(s) to provider_asset_market
Batch processing complete!
